In [ ]:
from importlib import import_module
import os
import sys
import argparse
import linecache
import uproot
import vector
import math
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
from tqdm import tqdm  # ✅ progress display
import glob
import json
vector.register_awkward()
import cmsstyle as CMS
import mplhep as hep
import matplotlib.pyplot as plt
from collections import Counter
sample = "/gv0/Users/achihwan/phase2/cmssw_16/condor/hltrun/new_150.root"
#sample = "/gv0/Users/achihwan/phase2/cmssw_16/condor/wjet/hltrun/wjet.root"
file = uproot.open(sample)
events = file["Events"]
runs = file["Runs"]
keys = events.keys()

In [19]:



tau_pt   = events["hltHpsPFTau_pt"].array()
tau_eta  = events["hltHpsPFTau_eta"].array()
tau_phi  = events["hltHpsPFTau_phi"].array()
tau_mass = events["hltHpsPFTau_mass"].array()

# Decay mode: based on GenVisTau_status (not reco tau DM)
genvis_tau_dm = events["GenVisTau_status"].array()

genvis_pt   = events["GenVisTau_pt"].array()
genvis_eta  = events["GenVisTau_eta"].array()
genvis_phi  = events["GenVisTau_phi"].array()
genvis_mass = events["GenVisTau_mass"].array()

genjet_pt  = events["GenJet_pt"].array()
genjet_eta = events["GenJet_eta"].array()
genjet_phi = events["GenJet_phi"].array()


pt_and_eta_cut_denominator = genvis_pt[(ak.num(genvis_pt)==1) & (abs(genvis_eta)<2.1) & (genvis_pt>130)]
print("denominator");print(len(ak.flatten(pt_and_eta_cut_denominator))); print((pt_and_eta_cut_denominator))
denominator = pt_and_eta_cut_denominator 


tau_trigger_filter = events["HLT_LooseDeepTauPFTauHPS180_L2NN_eta2p1"].array()
print(len(tau_trigger_filter))
trigger_passed = tau_trigger_filter ==True
print(len(tau_trigger_filter[trigger_passed]))

print(ak.num(genvis_pt))

counter = Counter(ak.to_numpy(ak.num(genvis_pt)).tolist())
for k, v in sorted(counter.items()):
    print(f"num={k}: {v} events","of total event")
print("-----")
counter = Counter(ak.to_numpy(ak.num(genvis_pt[trigger_passed])).tolist())
for k, v in sorted(counter.items()):
    print(f"num={k}: {v} events of tirgger passed")

mask1 = ak.num(genvis_pt[trigger_passed])==0
trigger_pass_no_gen = trigger_passed[mask1]
print(len(trigger_pass_no_gen ))

mask2 = ak.num(genvis_pt[trigger_passed])==1
trigger_pass_one_gen =  trigger_passed[ak.num(genvis_pt[trigger_passed])==1]
print(len(trigger_pass_one_gen ))

mask3 = ak.num(genvis_pt[trigger_passed])==2
trigger_pass_two_gen =  trigger_passed[mask3]
print(len(trigger_pass_two_gen ))

print("---")

print("total event",len(tau_pt))
trigger_pass_recotau = tau_pt[trigger_passed]
print("recotau,pass,trigger",len(trigger_pass_recotau))

trigger_pass_no_gen_recotau = trigger_pass_recotau[mask1]
print("no gen reco tau",len(trigger_pass_no_gen_recotau))

trigger_pass_one_gen_recotau = trigger_pass_recotau[mask2]
print("one gen reco tau",len(trigger_pass_one_gen_recotau))

trigger_pass_two_gen_recotau = trigger_pass_recotau[mask3]
print("two gen reco tau",len(trigger_pass_two_gen_recotau))
print ( " ----")


sorted_idx = ak.argsort(trigger_pass_no_gen_recotau, axis=1, ascending=False)
reco_pt_sorted = trigger_pass_no_gen_recotau[sorted_idx]
print(reco_pt_sorted[:,0])

denominator
3177
[[], [], [], [], [], [], [218], [146], ..., [242], [], [], [], [], [], [226]]
20000
5597
[2, 1, 0, 1, 1, 2, 1, 1, 2, 1, 2, 2, 1, ..., 1, 1, 2, 0, 0, 1, 1, 2, 2, 1, 2, 1]
num=0: 2805 events of total event
num=1: 9195 events of total event
num=2: 7997 events of total event
num=3: 3 events of total event
-----
num=0: 385 events of tirgger passed
num=1: 2328 events of tirgger passed
num=2: 2884 events of tirgger passed
385
2328
2884
---
total event 20000
recotau,pass,trigger 5597
no gen reco tau 385
one gen reco tau 2328
two gen reco tau 2884
 ----
[151, 170, 195, 183, 152, 185, 269, 159, ..., 152, 202, 260, 209, 196, 248, 246]


In [ ]:
#### Case: dR matching with reco pT > 130 cut; gen pT > 130 cut NOT applied



pt_and_eta_cut_denominator = genvis_pt[(ak.num(genvis_pt)==1) & (abs(genvis_eta)<2.1) ]#& (genvis_pt>130)]
pt_and_eta_cut_denominator_eta = genvis_eta[(ak.num(genvis_pt)==1) & (abs(genvis_eta)<2.1) ]#& (genvis_pt>130)]
print("denominator");print(len(ak.flatten(pt_and_eta_cut_denominator))); print((pt_and_eta_cut_denominator))
denominator = pt_and_eta_cut_denominator 
denominator_eta = pt_and_eta_cut_denominator_eta

gen_pt_1   = genvis_pt[trigger_passed][mask2] 
gen_eta_1  = genvis_eta[trigger_passed][mask2]
gen_phi_1  = genvis_phi[trigger_passed][mask2]
gen_dm_1   = genvis_tau_dm[trigger_passed][mask2]

reco_pt_1  = tau_pt[trigger_passed][mask2]       # shape: (N_events, var)
reco_eta_1 = tau_eta[trigger_passed][mask2]
reco_phi_1 = tau_phi[trigger_passed][mask2]
reco_mass_1= tau_mass[trigger_passed][mask2]
# exactly 1 gen tau per event → flatten to (N,) scalar-like for broadcasting
gen_eta_flat  = ak.flatten(gen_eta_1)   # (N,)  - safe since exactly 1 gen per event
gen_phi_flat  = ak.flatten(gen_phi_1)
gen_pt_flat   = ak.flatten(gen_pt_1)
gen_dm_flat   = ak.flatten(gen_dm_1)

pt_mask = reco_pt_1 > 130   # save mask before modifying reco arrays

reco_eta_1  = reco_eta_1[pt_mask]
reco_phi_1  = reco_phi_1[pt_mask]
reco_mass_1 = reco_mass_1[pt_mask]
reco_pt_1   = reco_pt_1[pt_mask]
print(gen_pt_flat)


# keep reco as (N, var)
# broadcast gen to match reco length
gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(gen_eta_flat[:, np.newaxis], reco_eta_1)
gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(gen_phi_flat[:, np.newaxis], reco_phi_1)

# deltaR calculation
deta = gen_eta_bc - reco_eta_bc
dphi = gen_phi_bc - reco_phi_bc
dphi = (dphi + np.pi) % (2 * np.pi) - np.pi

deltaR = np.sqrt(deta**2 + dphi**2)   # shape: (N, var)

# minimum deltaR per event
best_dR  = ak.min(deltaR, axis=1)          # (N,)
best_idx = ak.argmin(deltaR, axis=1, keepdims=True)  # (N, 1)

# matching cut
DR_CUT = 0.1

matched_mask = best_dR < DR_CUT

#########################################################
## gen matching
#########################################################
print(f"Number of Gen=1 events:         {len(gen_pt_flat)}")
print(f"Events matched (deltaR < {DR_CUT}):  {ak.sum(matched_mask)}")
print(f"Matching failed (no reco match): {ak.sum(~matched_mask)}")

# extract kinematics of matched events
matched_reco_pt  = reco_pt_1[matched_mask][best_idx[matched_mask]]
matched_reco_pt  = ak.flatten(matched_reco_pt)

matched_gen_pt   = gen_pt_flat[matched_mask]
matched_gen_dm   = gen_dm_flat[matched_mask]
matched_best_dR  = best_dR[matched_mask]
matched_gen_eta = gen_eta_flat[matched_mask]

print((matched_gen_pt))
print((matched_reco_pt))
print(matched_best_dR)


pt_eta_cut =  (np.abs(matched_gen_eta )<2.1) # & (matched_gen_pt > 130)
print("ptetacut length",len(pt_eta_cut))

after_gen_sel_gen_pt = matched_gen_pt[pt_eta_cut]
after_gen_sel_gen_eta = matched_gen_eta[pt_eta_cut]
print(len(after_gen_sel_gen_pt))
print(after_gen_sel_gen_pt)

after_gen_sel_recopt = matched_reco_pt[pt_eta_cut]
print(len(after_gen_sel_recopt))
print(after_gen_sel_recopt)

after_gen_sel_dr = matched_best_dR[pt_eta_cut]
print(len(after_gen_sel_dr))
print(after_gen_sel_dr)


denominator_using = ak.flatten(denominator)
print(denominator_using)
print("denominator length",len(denominator_using))
numerator = after_gen_sel_gen_pt
numerator_eta = after_gen_sel_gen_eta
print(numerator)
print("numerator length",len(numerator))

bins_0 = np.arange(0, 130, 10)
bins_1 = np.arange(131, 180, 5)
bins_2 = np.arange(181, 300, 20)
bins_3 = np.arange(301, 601, 100)
bins = np.concatenate((bins_0, bins_1, bins_2, bins_3))


pt_and_eta_cut_denominator_eta = genvis_eta[(ak.num(genvis_pt)==1) & (abs(genvis_eta)<2.1) & (genvis_pt>130)]
denominator_eta = pt_and_eta_cut_denominator_eta
denominator_using_eta = ak.flatten(denominator_eta)


numerator = numerator.to_numpy()
numerator_eta = numerator_eta.to_numpy()
denominator = denominator_using.to_numpy()
denominator_eta = denominator_using_eta.to_numpy()

# histogram counts
num_counts, num_edges = np.histogram(numerator, bins=bins)
den_counts, den_edges = np.histogram(denominator, bins=bins)


print(num_counts)
print(num_edges)

# bin centers
bin_centers = 0.5 * (num_edges[:-1] + num_edges[1:])
bin_widths = num_edges[1:] - num_edges[:-1]

numerator_yvalue =(num_counts/bin_widths)
denominator_yvalue =(den_counts/bin_widths)

plt.style.use(hep.style.CMS)
fig, ax = plt.subplots(figsize=(10, 8))

ax.hist(bin_centers, bins=num_edges, weights=numerator_yvalue,label='Numerator (GenVisTau pt)', color='C0',alpha=0.5)
ax.hist(bin_centers, bins=num_edges, weights=denominator_yvalue,label='Denominator (GenVisTau pt)', color='C1',alpha=0.5)
ax.set_xlabel(r'GenVisTau $p_T$ [GeV]', fontsize=18)
ax.set_ylabel('Events/GeV', fontsize=14)
ax.legend()
hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
ax.text(0.7, 1.06, r"$Z'(500GeV) \to \tau\tau$, PU=200",
        transform=ax.transAxes, fontsize=15, va='top', ha='left',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.tight_layout()
plt.show()




#### eta numerator / denominator histogram


bins_eta = np.arange(-2.5, 2.5, 0.1)
bin_centers_eta = 0.5 * (bins_eta[:-1] + bins_eta[1:])
bin_widths = bins_eta[1:] - bins_eta[:-1]


num_counts_eta, num_edges_eta = np.histogram(numerator_eta, bins=bins_eta)
den_counts_eta, den_edges_eta = np.histogram(denominator_eta, bins=bins_eta)

numerator_eta_yvalue =(num_counts_eta/bin_widths)
denominator_eta_yvalue =(den_counts_eta/bin_widths)

plt.style.use(hep.style.CMS)
fig, ax = plt.subplots(figsize=(10, 8))

ax.hist(bin_centers_eta, bins=num_edges_eta, weights=numerator_eta_yvalue,
        label='Numerator (GenVisTau eta)', color='C0', alpha=0.5)
ax.hist(bin_centers_eta, bins=num_edges_eta, weights=denominator_eta_yvalue,
        label='Denominator (GenVisTau eta)', color='C1', alpha=0.5)

ax.set_xlabel(r'GenVisTau $\eta$', fontsize=18)
ax.set_ylabel('Events/bin', fontsize=14)
ax.legend()

hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
ax.text(0.7, 1.06, r"$Z'(500GeV) \to \tau\tau$, PU=200",
        transform=ax.transAxes, fontsize=15, va='top', ha='left',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

plt.tight_layout()
plt.show()


counts_num_150, edges_pt = np.histogram(numerator, bins=bins)
counts_den_150, _ = np.histogram(denominator, bins=bins)

ratio_150 = np.divide(counts_num_150, counts_den_150, 
                      out=np.zeros_like(counts_num_150, dtype=float), 
                      where=(counts_den_150 != 0))
ratio_err_150 = np.zeros_like(ratio_150)
nonzero_150 = counts_den_150 > 0
ratio_err_150[nonzero_150] = np.sqrt(ratio_150[nonzero_150] * (1.0 - ratio_150[nonzero_150]) / counts_den_150[nonzero_150])

print(len(numerator)/len(denominator) *100 ,"efficiency of reco tau(%)")
bin_centers = (edges_pt[:-1] + edges_pt[1:]) / 2.0
half_widths = (edges_pt[1:] - edges_pt[:-1]) / 2.0

plt.style.use(hep.style.CMS)
fig, ax = plt.subplots(figsize=(10, 8))

# 1. Reco data
ax.errorbar(bin_centers, ratio_150, xerr=half_widths, yerr=ratio_err_150, 
            fmt='x', color='C0', ecolor='C0', capsize=4, alpha=0.8,
            label=r'Gen $\tau$')

# axis and label settings
ax.set_xlabel(r'GenVis Tau $p_T$ [GeV]', fontsize=18)
ax.set_ylabel('Efficiency', fontsize=18)
ax.set_xlim(edges_pt[0], edges_pt[-1])
ax.set_ylim(0, 1.1)

# trigger threshold line
ax.axvline(x=150, color='gray', linestyle='--', linewidth=2, label='Threshold (150 GeV)')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
# legend settings
ax.legend(loc='lower right', fontsize=14, frameon=True)

# grid and style
ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)

# CMS label
hep.cms.text("Simulation Preliminary", loc=0, ax=ax) # loc=0 = upper left
ax.text(0.7, 1.06, r"$Z'(500GeV) \to \tau \tau$, PU=200",
        transform=ax.transAxes, fontsize=15, va='top', ha='left',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

plt.tight_layout()
plt.show()

print(numerator)
print(denominator)
overthreshold_num_150 = numerator[numerator >= 150]
overthreshold_den_150 = denominator[denominator >= 150]
print("reco tau efficiency over 150GeV:", len(overthreshold_num_150)/len(overthreshold_den_150) *100 ,"%")

# ── eta efficiency (genvis pT > 130 cut) ─────────────────────────────────────
gen_eta_for_eff = np.asarray(matched_gen_eta[pt_eta_cut & (matched_gen_pt > 150)])
denom_eta = np.asarray(ak.flatten(genvis_eta[(ak.num(genvis_pt)==1) & (abs(genvis_eta)<2.1) & (genvis_pt>150)]))

bins_eta = np.linspace(-2.5, 2.5, 26)

counts_num_eta, edges_eta = np.histogram(gen_eta_for_eff, bins=bins_eta)
counts_den_eta, _         = np.histogram(denom_eta, bins=bins_eta)

ratio_eta     = np.divide(counts_num_eta, counts_den_eta,
                          out=np.zeros_like(counts_num_eta, dtype=float),
                          where=(counts_den_eta != 0))
nonzero_eta   = counts_den_eta > 0
ratio_err_eta = np.zeros_like(ratio_eta)
ratio_err_eta[nonzero_eta] = np.sqrt(
    ratio_eta[nonzero_eta] * (1 - ratio_eta[nonzero_eta]) / counts_den_eta[nonzero_eta])

bc_eta = (edges_eta[:-1] + edges_eta[1:]) / 2.0
hw_eta = (edges_eta[1:]  - edges_eta[:-1]) / 2.0

fig, ax = plt.subplots(figsize=(10, 8))
ax.errorbar(bc_eta, ratio_eta, xerr=hw_eta, yerr=ratio_err_eta,
            fmt='x', color='C1', ecolor='C1', capsize=4, alpha=0.8,
            label=r'Gen $\tau$')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel(r'GenVis Tau $\eta$', fontsize=18)
ax.set_ylabel('Efficiency', fontsize=18)
ax.set_xlim(edges_eta[0], edges_eta[-1])
ax.set_ylim(0, 1.1)
ax.legend(loc='lower center', fontsize=14, frameon=True)
ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
ax.text(0.7, 1.06, r"$Z'(500GeV) \to \tau\tau$, PU=200",
        transform=ax.transAxes, fontsize=15, va='top', ha='left',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.tight_layout()
plt.show()


# ── phi efficiency (genvis pT > 130 cut) ─────────────────────────────────────
gen_phi_for_eff = np.asarray(gen_phi_flat[matched_mask][pt_eta_cut & (matched_gen_pt > 150)])
denom_phi = np.asarray(ak.flatten(genvis_phi[(ak.num(genvis_pt)==1) & (abs(genvis_eta)<2.1) & (genvis_pt>150)]))

bins_phi = np.linspace(-np.pi, np.pi, 26)

counts_num_phi, edges_phi = np.histogram(gen_phi_for_eff, bins=bins_phi)
counts_den_phi, _         = np.histogram(denom_phi, bins=bins_phi)

ratio_phi     = np.divide(counts_num_phi, counts_den_phi,
                          out=np.zeros_like(counts_num_phi, dtype=float),
                          where=(counts_den_phi != 0))
nonzero_phi   = counts_den_phi > 0
ratio_err_phi = np.zeros_like(ratio_phi)
ratio_err_phi[nonzero_phi] = np.sqrt(
    ratio_phi[nonzero_phi] * (1 - ratio_phi[nonzero_phi]) / counts_den_phi[nonzero_phi])

bc_phi = (edges_phi[:-1] + edges_phi[1:]) / 2.0
hw_phi = (edges_phi[1:]  - edges_phi[:-1]) / 2.0

fig, ax = plt.subplots(figsize=(10, 8))
ax.errorbar(bc_phi, ratio_phi, xerr=hw_phi, yerr=ratio_err_phi,
            fmt='x', color='C2', ecolor='C2', capsize=4, alpha=0.8,
            label=r'Gen $\tau$')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel(r'GenVis Tau $\phi$', fontsize=18)
ax.set_ylabel('Efficiency', fontsize=18)
ax.set_xlim(edges_phi[0], edges_phi[-1])
ax.set_ylim(0, 1.1)
ax.legend(loc='lower center', fontsize=14, frameon=True)
ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
ax.text(0.7, 1.06, r"$Z'(500GeV) \to \tau\tau$, PU=200",
        transform=ax.transAxes, fontsize=15, va='top', ha='left',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.tight_layout()
plt.show()


N_num = len(overthreshold_num_150)
N_den = len(overthreshold_den_150)

eff = N_num / N_den
eff_err = np.sqrt(eff * (1 - eff) / N_den)

print(f"Efficiency: {eff*100:.2f} ± {eff_err*100:.2f} %")

In [ ]:
"""
HPS Tau Trigger Efficiency Analysis
- Fraction of gen-matched reco taus passing HLT (Trigger Turn-on)
- Efficiency and num/den distribution plots vs. GenVisTau pT / eta / phi
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
import uproot
import mplhep as hep

# ══════════════════════════════════════════════════════════════════
# 0. Constants
# ══════════════════════════════════════════════════════════════════
HEADER   = r"$Z'(500\,\mathrm{GeV})\to\tau\tau$, PU=200"
PT_THR   = 150        # plateau threshold pT [GeV]
DR_CUT   = 0.1        # gen-matching dR threshold
ETA_CUT  = 2.1        # |eta| cut
RECO_PT_CUT = 130.0   # leading reco tau pT cut [GeV]

_b0 = np.arange(0,   130, 10)
_b1 = np.arange(131, 180,  5)
_b2 = np.arange(181, 300, 20)
_b3 = np.arange(301, 601, 100)
BINS_PT  = np.concatenate((_b0, _b1, _b2, _b3))
BINS_ETA = np.linspace(-2.5, 2.5,  26)
BINS_PHI = np.linspace(-np.pi, np.pi, 26)


# ══════════════════════════════════════════════════════════════════
# 1. Utilities
# ══════════════════════════════════════════════════════════════════
def ratio_and_err(num, den):
    r = np.divide(num, den,
                  out=np.zeros_like(num, dtype=float),
                  where=(den != 0))
    e = np.where(den > 0,
                 np.sqrt(r * (1 - r) / np.maximum(den, 1)), 0.0)
    return r, e


def _save_or_show(fig, save_dir, filename):
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, filename), dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  saved → {filename}")
    else:
        plt.show()
        plt.close(fig)


def _cms_label(ax):
    hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
    ax.text(0.97, 1.02, HEADER,
            transform=ax.transAxes, fontsize=11,
            va="bottom", ha="right",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))


# ══════════════════════════════════════════════════════════════════
# 2. Data Loading
# ══════════════════════════════════════════════════════════════════
def load_data(sample):
    events = uproot.open(sample)["Events"]

    tau_pt   = events["hltHpsPFTau_pt"].array()
    tau_eta  = events["hltHpsPFTau_eta"].array()
    tau_phi  = events["hltHpsPFTau_phi"].array()

    gv_pt    = events["GenVisTau_pt"].array()
    gv_eta   = events["GenVisTau_eta"].array()
    gv_phi   = events["GenVisTau_phi"].array()

    trigger  = events["HLT_LooseDeepTauPFTauHPS180_L2NN_eta2p1"].array()

    # Event filter: reco tau >= 1 AND GenVisTau == 1
    evt_mask = np.asarray((ak.num(tau_pt) >= 1) & (ak.num(gv_pt) == 1))

    tau_pt  = tau_pt[evt_mask];  tau_eta = tau_eta[evt_mask]
    tau_phi = tau_phi[evt_mask]
    gv_pt   = gv_pt[evt_mask];   gv_eta  = gv_eta[evt_mask]
    gv_phi  = gv_phi[evt_mask]
    trigger = trigger[evt_mask]

    print(f"Events after filter (reco>=1, GenVisTau==1): {evt_mask.sum()}")
    return tau_pt, tau_eta, tau_phi, gv_pt, gv_eta, gv_phi, trigger


# ══════════════════════════════════════════════════════════════════
# 3. Gen Matching  (for efficiency: matched = REAL tau)
#    leading reco tau (pT > RECO_PT_CUT, |eta| < ETA_CUT)  ↔  GenVisTau
#    dR < DR_CUT  →  matched = True
# ══════════════════════════════════════════════════════════════════
def gen_match(tau_pt, tau_eta, tau_phi, gv_eta, gv_phi):
    # extract leading reco tau
    lead_idx = ak.argmax(tau_pt, axis=1, keepdims=True)
    lead_pt  = np.asarray(ak.flatten(tau_pt[lead_idx]))   # (N,)
    lead_eta = np.asarray(ak.flatten(tau_eta[lead_idx]))
    lead_phi = np.asarray(ak.flatten(tau_phi[lead_idx]))

    # exactly 1 GenVisTau per event → flatten
    gv_eta_1d = np.asarray(ak.flatten(gv_eta))            # (N,)
    gv_phi_1d = np.asarray(ak.flatten(gv_phi))

    # compute dR
    deta = lead_eta - gv_eta_1d
    dphi = (lead_phi - gv_phi_1d + np.pi) % (2 * np.pi) - np.pi
    dR   = np.sqrt(deta**2 + dphi**2)

    matched = (lead_pt > RECO_PT_CUT) & (np.abs(lead_eta) < ETA_CUT) & (dR < DR_CUT)
    print(f"Gen-matched events: {matched.sum()} / {len(matched)}")
    return matched   # shape (N,), bool


# ══════════════════════════════════════════════════════════════════
# 4. Denominator / Numerator Array Construction
#    Den : |GenVisTau eta| < ETA_CUT
#    Num : Den  ∩  gen-matched  ∩  trigger passed
# ══════════════════════════════════════════════════════════════════
def build_arrays(gv_pt, gv_eta, gv_phi, trigger, matched):
    # 1 GenVisTau per event → flatten to (N,) 1D arrays
    gv_pt_1d  = np.asarray(ak.flatten(gv_pt))
    gv_eta_1d = np.asarray(ak.flatten(gv_eta))
    gv_phi_1d = np.asarray(ak.flatten(gv_phi))
    trig      = np.asarray(trigger, dtype=bool)

    # masks (all shape (N,))
    den_mask = np.abs(gv_eta_1d) < ETA_CUT
    num_mask = den_mask & matched & trig

    print(f"\nDenominator (|eta|<{ETA_CUT}):  {den_mask.sum()}")
    print(f"Numerator (matched & triggered): {num_mask.sum()}")

    # print plateau value
    n_d = (gv_pt_1d[den_mask] >= PT_THR).sum()
    n_n = (gv_pt_1d[num_mask] >= PT_THR).sum()
    eff = n_n / n_d if n_d else 0.0
    err = np.sqrt(eff * (1 - eff) / max(n_d, 1))
    print(f"Plateau efficiency (pT>{PT_THR} GeV): "
          f"{eff*100:.2f} ± {err*100:.2f}%  ({n_n}/{n_d})\n")

    # full denominator/numerator arrays (order guaranteed to match)
    den = dict(pt=gv_pt_1d[den_mask],
               eta=gv_eta_1d[den_mask],
               phi=gv_phi_1d[den_mask])
    num = dict(pt=gv_pt_1d[num_mask],
               eta=gv_eta_1d[num_mask],
               phi=gv_phi_1d[num_mask])
    return num, den


# ══════════════════════════════════════════════════════════════════
# 5. Num / Den Histograms (step + error bar style)
# ══════════════════════════════════════════════════════════════════
DEN_COLOR = "#4C78A8"
NUM_COLOR = "#E45756"


def _hist_step_normed(ax, data, bins, color, label, linestyle="-"):
    counts, edges = np.histogram(data, bins=bins)
    widths  = edges[1:] - edges[:-1]
    heights = counts / widths
    ax.step(edges[:-1], heights, where="post",
            color=color, linestyle=linestyle,
            linewidth=2.0, alpha=0.9, label=label)
    err = np.sqrt(counts) / widths
    bc  = (edges[:-1] + edges[1:]) / 2
    ax.errorbar(bc, heights, yerr=err,
                fmt="none", color=color, capsize=2, alpha=0.7)


def plot_num_den(num_arr, den_arr, bins, xlabel,
                 pt_cut=None, filename=None, save_dir=None):
    plt.style.use(hep.style.CMS)
    fig, ax = plt.subplots(figsize=(10, 7))

    _hist_step_normed(ax, den_arr, bins, DEN_COLOR, "Denominator", "--")
    _hist_step_normed(ax, num_arr, bins, NUM_COLOR, "Numerator",   "-")

    if pt_cut is not None:
        ax.axvline(x=pt_cut, color="gray", linestyle="--",
                   linewidth=1.5, label=f"pT cut ({pt_cut} GeV)")

    ax.set_xlabel(xlabel,            fontsize=16)
    ax.set_ylabel("Events / bin width", fontsize=16)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=12, loc="upper right", frameon=True)
    ax.grid(which="both", linestyle=":", linewidth=0.5, alpha=0.5)
    _cms_label(ax)
    plt.tight_layout()
    _save_or_show(fig, save_dir, filename)


# ══════════════════════════════════════════════════════════════════
# 6. Efficiency Plots
# ══════════════════════════════════════════════════════════════════
def plot_efficiency(num_arr, den_arr, bins, xlabel,
                    color="C0", vline=None, legend_loc="lower right",
                    filename=None, save_dir=None):
    c_n, edges = np.histogram(num_arr, bins=bins)
    c_d, _     = np.histogram(den_arr, bins=bins)
    r, e = ratio_and_err(c_n, c_d)
    bc = (edges[:-1] + edges[1:]) / 2
    hw = (edges[1:]  - edges[:-1]) / 2

    plt.style.use(hep.style.CMS)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.errorbar(bc, r, xerr=hw, yerr=e,
                fmt="o", color=color, ecolor=color,
                capsize=3, alpha=0.85, label=r"Gen $\tau$")
    if vline is not None:
        ax.axvline(x=vline, color="gray", linestyle="--",
                   linewidth=2, label=f"Threshold ({vline} GeV)")
    ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel(xlabel,      fontsize=16)
    ax.set_ylabel("Efficiency", fontsize=16)
    ax.set_xlim(edges[0], edges[-1])
    ax.set_ylim(0, 1.15)
    ax.legend(loc=legend_loc, fontsize=12, frameon=True)
    ax.grid(which="both", linestyle=":", linewidth=0.5, alpha=0.5)
    _cms_label(ax)
    plt.tight_layout()
    _save_or_show(fig, save_dir, filename)


# ══════════════════════════════════════════════════════════════════
# 7. Main
# ══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    SAMPLE   = "/gv0/Users/achihwan/phase2/cmssw_16/condor/hltrun/new_150.root"
    SAVE_DIR = "./efficiency_plots/"

    # ── 1) Load data ─────────────────────────────────────────────
    tau_pt, tau_eta, tau_phi, gv_pt, gv_eta, gv_phi, trigger = load_data(SAMPLE)

    # ── 2) Gen matching ──────────────────────────────────────────
    matched = gen_match(tau_pt, tau_eta, tau_phi, gv_eta, gv_phi)

    # ── 3) Build Den / Num arrays ────────────────────────────────
    num, den = build_arrays(gv_pt, gv_eta, gv_phi, trigger, matched)

    # apply additional pT > PT_THR cut for eta / phi
    thr_d = den["pt"] >= PT_THR
    thr_n = num["pt"] >= PT_THR

    # ── 4) Num / Den Histograms ──────────────────────────────────
    plot_num_den(num["pt"],         den["pt"],
            BINS_PT,  r"GenVis Tau $p_T$ [GeV]",
            pt_cut=PT_THR,                          # ← add vertical threshold line
            filename="numden_pt.png",  save_dir=SAVE_DIR)

    plot_num_den(num["eta"][thr_n], den["eta"][thr_d],
            BINS_ETA, rf"GenVis Tau $\eta$  ($p_T>{PT_THR}$ GeV)",
            filename="numden_eta.png", save_dir=SAVE_DIR)

    plot_num_den(num["phi"][thr_n], den["phi"][thr_d],
            BINS_PHI, rf"GenVis Tau $\phi$  ($p_T>{PT_THR}$ GeV)",
            filename="numden_phi.png", save_dir=SAVE_DIR)

    # ── 5) Efficiency Plots ──────────────────────────────────────
    plot_efficiency(num["pt"],         den["pt"],
                    BINS_PT, r"GenVis Tau $p_T$ [GeV]",
                    color="C0", vline=PT_THR, legend_loc="lower right",
                    filename="efficiency_pt.png",  save_dir=SAVE_DIR)

    plot_efficiency(num["eta"][thr_n], den["eta"][thr_d],
                    BINS_ETA, rf"GenVis Tau $\eta$  ($p_T>{PT_THR}$ GeV)",
                    color="C1", legend_loc="lower center",
                    filename="efficiency_eta.png", save_dir=SAVE_DIR)

    plot_efficiency(num["phi"][thr_n], den["phi"][thr_d],
                    BINS_PHI, rf"GenVis Tau $\phi$  ($p_T>{PT_THR}$ GeV)",
                    color="C2", legend_loc="lower center",
                    filename="efficiency_phi.png", save_dir=SAVE_DIR)

In [ ]:
from importlib import import_module
import os
import sys
import argparse
import linecache
import uproot
import vector
import math
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
from tqdm import tqdm  # ✅ progress display
import glob
import json
vector.register_awkward()
import cmsstyle as CMS
import mplhep as hep
import matplotlib.pyplot as plt
from collections import Counter
sample = "/gv0/Users/achihwan/phase2/cmssw_16/condor/hltrun/new_150.root"
file = uproot.open(sample)
events = file["Events"]
runs = file["Runs"]
keys = events.keys()



tau_pt   = events["hltHpsPFTau_pt"].array()
tau_eta  = events["hltHpsPFTau_eta"].array()
tau_phi  = events["hltHpsPFTau_phi"].array()
tau_mass = events["hltHpsPFTau_mass"].array()

# Decay mode: based on GenVisTau_status (not reco tau DM)
genvis_tau_dm = events["GenVisTau_status"].array()

genvis_pt   = events["GenVisTau_pt"].array()
genvis_eta  = events["GenVisTau_eta"].array()
genvis_phi  = events["GenVisTau_phi"].array()
genvis_mass = events["GenVisTau_mass"].array()

genjet_pt  = events["GenJet_pt"].array()
genjet_eta = events["GenJet_eta"].array()
genjet_phi = events["GenJet_phi"].array()



tau_trigger_filter = events["HLT_LooseDeepTauPFTauHPS180_L2NN_eta2p1"].array()

def analyze_tau_matching(
    reco_pt_sel, reco_eta_sel, reco_phi_sel,
    gen_pt_sel, gen_eta_sel, gen_phi_sel, gen_dm_sel,
    DR_CUT=0.3,
    label="",
    gen_pt_cut=130, gen_eta_cut=2.1, reco_pt_cut=130
):
    """
    Performs matching analysis given a single gen tau (N, 1) or pre-flattened (N,) gen arrays
    and reco tau (N, var) arrays.
    gen_pt_sel, gen_eta_sel, etc. must be pre-flattened to (N,) before calling.
    """
    # deltaR computation
    gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(gen_eta_sel[:, np.newaxis], reco_eta_sel)
    gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(gen_phi_sel[:, np.newaxis], reco_phi_sel)

    deta = gen_eta_bc - reco_eta_bc
    dphi = gen_phi_bc - reco_phi_bc
    dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
    deltaR = np.sqrt(deta**2 + dphi**2)

    best_dR  = ak.min(deltaR, axis=1)
    best_idx = ak.argmin(deltaR, axis=1, keepdims=True)
    matched_mask = best_dR < DR_CUT

    # matched kinematics
    matched_reco_pt  = ak.flatten(reco_pt_sel[matched_mask][best_idx[matched_mask]])
    matched_gen_pt   = gen_pt_sel[matched_mask]
    matched_gen_eta  = gen_eta_sel[matched_mask]
    matched_gen_dm   = gen_dm_sel[matched_mask]
    matched_best_dR  = best_dR[matched_mask]

    # gen pt/eta cut
    pt_eta_cut = (matched_gen_pt > gen_pt_cut) & (np.abs(matched_gen_eta) < gen_eta_cut)

    after_gen_sel_gen_pt  = matched_gen_pt[pt_eta_cut]
    after_gen_sel_reco_pt = matched_reco_pt[pt_eta_cut]
    after_gen_sel_dr      = matched_best_dR[pt_eta_cut]

    # full reco list (for leading reco pt)
    reco_pt_after_cut = reco_pt_sel[matched_mask][pt_eta_cut]
    sorted_idx    = ak.argsort(reco_pt_after_cut, axis=1, ascending=False)
    leading_reco  = reco_pt_after_cut[sorted_idx][:, 0]

    # reco pt > reco_pt_cut (L1 cut)
    pt130_filter = after_gen_sel_reco_pt > reco_pt_cut

    # case: matched reco pt > 130
    lead_reco_pass     = leading_reco[pt130_filter]
    matched_reco_pass  = after_gen_sel_reco_pt[pt130_filter]
    has_higher = lead_reco_pass > matched_reco_pass

    # case: matched reco pt < 130
    lead_reco_fail    = leading_reco[~pt130_filter]
    matched_reco_fail = after_gen_sel_reco_pt[~pt130_filter]
    has_higher_fail   = lead_reco_fail > matched_reco_fail

    # summary print
    print(f"\n{'='*60}")
    print(f"[{label}]  DR_CUT = {DR_CUT}")
    print(f"{'='*60}")
    print(f"  Number of input events                  : {len(gen_pt_sel)}")
    print(f"  Events matched (deltaR < {DR_CUT}):         {ak.sum(matched_mask)}")
    print(f"  Matching failed (no reco match):         {ak.sum(~matched_mask)}")
    print(f"  gen pt>{gen_pt_cut}, |eta|<{gen_eta_cut}:               {len(after_gen_sel_gen_pt)}")
    print(f"  └─ matched reco pt > {reco_pt_cut}:           {len(matched_reco_pass)}")
    print(f"     ├─ leading reco > matched (unknown):  {int(ak.sum(has_higher))}")
    print(f"     └─ matched is leading (triggered):    {len(matched_reco_pass) - int(ak.sum(has_higher))}")
    print(f"  └─ matched reco pt < {reco_pt_cut}:           {len(matched_reco_fail)}")
    print(f"     └─ has higher reco than matched:      {int(ak.sum(has_higher_fail))}")

    return {
        "matched_mask"      : matched_mask,
        "matched_gen_pt"    : matched_gen_pt,
        "matched_gen_eta"   : matched_gen_eta,
        "matched_gen_dm"    : matched_gen_dm,
        "matched_reco_pt"   : matched_reco_pt,
        "matched_best_dR"   : matched_best_dR,
        "after_gen_sel_gen_pt"  : after_gen_sel_gen_pt,
        "after_gen_sel_reco_pt" : after_gen_sel_reco_pt,
        "after_gen_sel_dr"      : after_gen_sel_dr,
        "leading_reco"      : leading_reco,
    }


def scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1,
                 gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5]):
    """gen=1 case: compare multiple DR_CUT values"""
    gen_pt_flat  = ak.flatten(gen_pt_1)
    gen_eta_flat = ak.flatten(gen_eta_1)
    gen_phi_flat = ak.flatten(gen_phi_1)
    gen_dm_flat  = ak.flatten(gen_dm_1)

    results = {}
    for dr in dr_list:
        res = analyze_tau_matching(
            reco_pt_1, reco_eta_1, reco_phi_1,
            gen_pt_flat, gen_eta_flat, gen_phi_flat, gen_dm_flat,
            DR_CUT=dr, label=f"Gen=1"
        )
        results[dr] = res
    return results


def scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2,
                 gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5]):
    """
    gen=2 case: try matching leading gen first, fall back to subleading on failure
    leading    = higher gen pT  (index 0 after argsort descending)
    subleading = lower gen pT   (index 1)
    """
    # split into leading / subleading
    gen_sort_idx   = ak.argsort(gen_pt_2, axis=1, ascending=False)
    gen_pt_sorted  = gen_pt_2[gen_sort_idx]
    gen_eta_sorted = gen_eta_2[gen_sort_idx]
    gen_phi_sorted = gen_phi_2[gen_sort_idx]
    gen_dm_sorted  = gen_dm_2[gen_sort_idx]

    # flatten to (N,)
    lead_pt  = ak.flatten(gen_pt_sorted[:, 0:1])
    lead_eta = ak.flatten(gen_eta_sorted[:, 0:1])
    lead_phi = ak.flatten(gen_phi_sorted[:, 0:1])
    lead_dm  = ak.flatten(gen_dm_sorted[:, 0:1])

    sub_pt   = ak.flatten(gen_pt_sorted[:, 1:2])
    sub_eta  = ak.flatten(gen_eta_sorted[:, 1:2])
    sub_phi  = ak.flatten(gen_phi_sorted[:, 1:2])
    sub_dm   = ak.flatten(gen_dm_sorted[:, 1:2])

    results = {}
    for dr in dr_list:
        print(f"\n{'#'*60}")
        print(f"  Gen=2  |  DR_CUT = {dr}")
        print(f"{'#'*60}")

        # compute leading gen deltaR
        gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(lead_eta[:, np.newaxis], reco_eta_2)
        gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(lead_phi[:, np.newaxis], reco_phi_2)
        deta = gen_eta_bc - reco_eta_bc
        dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
        deltaR_lead = np.sqrt(deta**2 + dphi**2)
        best_dR_lead = ak.min(deltaR_lead, axis=1)

        lead_matched_mask = best_dR_lead < dr
        lead_fail_mask    = ~lead_matched_mask

        print(f"\n  [Leading gen matched: {ak.sum(lead_matched_mask)} / {len(lead_pt)} events]")

        # analyze events where leading gen matched
        res_lead = analyze_tau_matching(
            reco_pt_2[lead_matched_mask],
            reco_eta_2[lead_matched_mask],
            reco_phi_2[lead_matched_mask],
            lead_pt[lead_matched_mask],
            lead_eta[lead_matched_mask],
            lead_phi[lead_matched_mask],
            lead_dm[lead_matched_mask],
            DR_CUT=dr, label="Gen=2 Leading matched"
        )

        # leading match failed → try subleading
        print(f"\n  [Leading match failed → Trying subleading: {ak.sum(lead_fail_mask)} events]")
        res_sub = analyze_tau_matching(
            reco_pt_2[lead_fail_mask],
            reco_eta_2[lead_fail_mask],
            reco_phi_2[lead_fail_mask],
            sub_pt[lead_fail_mask],
            sub_eta[lead_fail_mask],
            sub_phi[lead_fail_mask],
            sub_dm[lead_fail_mask],
            DR_CUT=dr, label="Gen=2 Subleading (leading failed)"
        )

        results[dr] = {"leading": res_lead, "subleading": res_sub}

    return results

In [ ]:
dr_list = [0.1]

# ─────────────────────────────────────────────
# Prepare GenVisTau, GenJet (η, φ) arrays aligned with Gen=1 / Gen=2 selection
# ─────────────────────────────────────────────

# Gen=1 (mask2) — re-defined here for clarity (gen_eta_1, gen_phi_1 were set earlier)
genvis_eta_1 = genvis_eta[trigger_passed][mask2]
genvis_phi_1 = genvis_phi[trigger_passed][mask2]
genjet_eta_1 = genjet_eta[trigger_passed][mask2]
genjet_phi_1 = genjet_phi[trigger_passed][mask2]

# Gen=2 (mask3)
genvis_eta_2 = genvis_eta[trigger_passed][mask3]
genvis_phi_2 = genvis_phi[trigger_passed][mask3]
genjet_eta_2 = genjet_eta[trigger_passed][mask3]
genjet_phi_2 = genjet_phi[trigger_passed][mask3]

# ─────────────────────────────────────────────
# isolated GenJet count per event as (N,) numpy int array
# ─────────────────────────────────────────────
njets_1 = count_isolated_jets(genvis_eta_1, genvis_phi_1,
                              genjet_eta_1, genjet_phi_1,
                              deltaR_threshold=0.3)

njets_2 = count_isolated_jets(genvis_eta_2, genvis_phi_2,
                              genjet_eta_2, genjet_phi_2,
                              deltaR_threshold=0.3)

# length sanity check (mismatch would silently corrupt broadcasting/masking)
assert len(njets_1) == len(gen_pt_1) == len(reco_pt_1), \
    f"gen=1 length mismatch: njets={len(njets_1)}, gen={len(gen_pt_1)}, reco={len(reco_pt_1)}"
assert len(njets_2) == len(gen_pt_2) == len(reco_pt_2), \
    f"gen=2 length mismatch: njets={len(njets_2)}, gen={len(gen_pt_2)}, reco={len(reco_pt_2)}"

# ─────────────────────────────────────────────
# run matching analysis
# ─────────────────────────────────────────────
'''
res1 = scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1,
                    gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                    njets_1,
                    dr_list=[0.1])

res2 = scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2,
                    gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                    njets_2,
                    dr_list=[0.1])

'''

In [ ]:
import numpy as np
import awkward as ak

# ═══════════════════════════════════════════════════════════
# 1. DM / Jet Group Definitions
# ═══════════════════════════════════════════════════════════
DM_GROUPS = {
    "1Prong_DM0":  [0],
    "1Prong_DM1":  [1],
    "1Prong_DM2":  [2],
    "3Prong_DM10": [10],
    "3Prong_DM11": [11],
    "1Prong_all":  [0, 1, 2],
    "3Prong_all":  [10, 11],
}
JET_GROUPS = {
    "0jet":   lambda n: n == 0,
    "1jet":   lambda n: n == 1,
    ">=2jet": lambda n: n >= 2,
}

# ═══════════════════════════════════════════════════════════
# 2. Helper Functions
# ═══════════════════════════════════════════════════════════
def _dR(eta1, phi1, eta2, phi2):
    deta = eta1 - eta2
    dphi = (phi1 - phi2 + np.pi) % (2 * np.pi) - np.pi
    return np.sqrt(deta**2 + dphi**2)


def count_isolated_jets(genvis_eta, genvis_phi,
                        genjet_eta, genjet_phi,
                        deltaR_threshold=0.3):
    """Returns per-event count of GenJets with ΔR ≥ threshold from all GenVisTau as (N,) array"""
    n_arr = []
    for i in range(len(genvis_eta)):
        g_eta_i = np.array(genvis_eta[i])
        g_phi_i = np.array(genvis_phi[i])
        jet_eta_i = np.array(genjet_eta[i])
        jet_phi_i = np.array(genjet_phi[i])
        count = 0
        for j in range(len(jet_eta_i)):
            j_eta = float(jet_eta_i[j]); j_phi = float(jet_phi_i[j])
            is_iso = all(
                _dR(float(g_eta_i[t]), float(g_phi_i[t]), j_eta, j_phi) >= deltaR_threshold
                for t in range(len(g_eta_i))
            )
            if is_iso:
                count += 1
        n_arr.append(count)
    return np.array(n_arr, dtype=int)


def print_dm_jet_table(label, dm_arr, njets_arr, indent="    "):
    dm_arr    = np.asarray(dm_arr,    dtype=int)
    njets_arr = np.asarray(njets_arr, dtype=int)
    total_n   = len(dm_arr)
    jet_keys  = list(JET_GROUPS.keys())
    col_w     = 12

    print(f"{indent}[{label}]  subset N = {total_n}")
    header = f"{indent}{'DM':<15}"
    for jk in jet_keys:
        header += f"{jk:>{col_w}}"
    header += f"{'total':>{col_w}}"
    print(header)
    print(f"{indent}{'-' * (15 + col_w * (len(jet_keys) + 1))}")

    if total_n == 0:
        print(f"{indent}(empty)")
        return

    for dk, modes in DM_GROUPS.items():
        dm_mask = np.zeros(total_n, dtype=bool)
        for m in modes:
            dm_mask |= (dm_arr == m)
        row = f"{indent}{dk:<15}"
        total = 0
        for jk in jet_keys:
            sel = JET_GROUPS[jk]
            cnt = int(np.sum(dm_mask & sel(njets_arr)))
            row += f"{cnt:>{col_w}}"
            total += cnt
        row += f"{total:>{col_w}}"
        print(row)


# ═══════════════════════════════════════════════════════════
# 3. Matching Analysis (with njets, DM×Jet table output)
# ═══════════════════════════════════════════════════════════
def analyze_tau_matching(
    reco_pt_sel, reco_eta_sel, reco_phi_sel,
    gen_pt_sel,  gen_eta_sel,  gen_phi_sel,  gen_dm_sel,
    njets_sel,
    DR_CUT=0.3,
    label="",
    gen_pt_cut=130, gen_eta_cut=2.1, reco_pt_cut=130,
):
    # ---- ΔR matching ----
    gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(gen_eta_sel[:, np.newaxis], reco_eta_sel)
    gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(gen_phi_sel[:, np.newaxis], reco_phi_sel)
    deta = gen_eta_bc - reco_eta_bc
    dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
    deltaR = np.sqrt(deta**2 + dphi**2)

    best_dR  = ak.min(deltaR, axis=1)
    best_idx = ak.argmin(deltaR, axis=1, keepdims=True)
    matched_mask = best_dR < DR_CUT

    matched_reco_pt  = ak.flatten(reco_pt_sel[matched_mask][best_idx[matched_mask]])
    matched_gen_pt   = gen_pt_sel[matched_mask]
    matched_gen_eta  = gen_eta_sel[matched_mask]
    matched_gen_dm   = gen_dm_sel[matched_mask]
    matched_best_dR  = best_dR[matched_mask]

    # ---- gen pt/η cut ----
    pt_eta_cut = (matched_gen_pt > gen_pt_cut) & (np.abs(matched_gen_eta) < gen_eta_cut)
    after_gen_sel_gen_pt  = matched_gen_pt[pt_eta_cut]
    after_gen_sel_reco_pt = matched_reco_pt[pt_eta_cut]
    after_gen_sel_dr      = matched_best_dR[pt_eta_cut]

    reco_pt_after_cut = reco_pt_sel[matched_mask][pt_eta_cut]
    sorted_idx        = ak.argsort(reco_pt_after_cut, axis=1, ascending=False)
    leading_reco      = reco_pt_after_cut[sorted_idx][:, 0]

    # ---- L1 cut: matched reco pt > 130 ----
    pt130_filter = after_gen_sel_reco_pt > reco_pt_cut
    lead_reco_pass    = leading_reco[pt130_filter]
    matched_reco_pass = after_gen_sel_reco_pt[pt130_filter]
    has_higher        = lead_reco_pass > matched_reco_pass

    lead_reco_fail    = leading_reco[~pt130_filter]
    matched_reco_fail = after_gen_sel_reco_pt[~pt130_filter]
    has_higher_fail   = lead_reco_fail > matched_reco_fail

    # ---- DM/njets numpy array cleanup ----
    def _tonp_bool(x):
        return ak.to_numpy(ak.fill_none(x, False)).astype(bool)

    njets_sel_np = np.asarray(njets_sel, dtype=int)
    dm_sel_np    = ak.to_numpy(gen_dm_sel).astype(int)

    mm_np            = _tonp_bool(matched_mask)
    dm_matched       = dm_sel_np[mm_np]
    njets_matched    = njets_sel_np[mm_np]

    pe_np            = _tonp_bool(pt_eta_cut)
    dm_after_cut     = dm_matched[pe_np]
    njets_after_cut  = njets_matched[pe_np]

    p130_np          = _tonp_bool(pt130_filter)
    dm_p130_pass     = dm_after_cut[p130_np]
    njets_p130_pass  = njets_after_cut[p130_np]
    dm_p130_fail     = dm_after_cut[~p130_np]
    njets_p130_fail  = njets_after_cut[~p130_np]

    hh_np  = _tonp_bool(has_higher)
    hhf_np = _tonp_bool(has_higher_fail)

    dm_lead_higher     = dm_p130_pass[hh_np]
    njets_lead_higher  = njets_p130_pass[hh_np]
    dm_matched_lead    = dm_p130_pass[~hh_np]
    njets_matched_lead = njets_p130_pass[~hh_np]
    dm_fail_higher     = dm_p130_fail[hhf_np]
    njets_fail_higher  = njets_p130_fail[hhf_np]

    # ---- print summary ----
    n_trig_total       = len(matched_reco_pass)
    n_trig_ambiguous   = int(ak.sum(has_higher))
    n_trig_unambiguous = n_trig_total - n_trig_ambiguous

    print(f"\n{'='*60}")
    print(f"[{label}]  DR_CUT = {DR_CUT}")
    print(f"{'='*60}")
    print(f"  Number of input events                  : {len(gen_pt_sel)}")

    print(f"  ΔR < {DR_CUT} matched                      : {ak.sum(matched_mask)}")
    print_dm_jet_table("matched", dm_matched, njets_matched)

    print(f"  Matching failed                         : {ak.sum(~matched_mask)}")

    print(f"  gen pt>{gen_pt_cut}, |η|<{gen_eta_cut}            : {len(after_gen_sel_gen_pt)}")
    print_dm_jet_table("passed gen pt/η cut", dm_after_cut, njets_after_cut)

    print(f"  └─ matched reco pt > {reco_pt_cut}  [TRIGGERED total] : {n_trig_total}")
    print_dm_jet_table(f"TRIGGERED (matched reco pt > {reco_pt_cut})",
                       dm_p130_pass, njets_p130_pass, indent="      ")

    print(f"     ├─ leading reco > matched (triggered, ambiguous) : {n_trig_ambiguous}")
    print_dm_jet_table("triggered, ambiguous (leading > matched)",
                       dm_lead_higher, njets_lead_higher, indent="        ")

    print(f"     └─ matched is leading (triggered, unambiguous)   : {n_trig_unambiguous}")
    print_dm_jet_table("triggered, unambiguous (matched is leading)",
                       dm_matched_lead, njets_matched_lead, indent="        ")

    print(f"  └─ matched reco pt < {reco_pt_cut}  [NOT triggered]   : {len(matched_reco_fail)}")
    print_dm_jet_table(f"NOT triggered (matched reco pt < {reco_pt_cut})",
                       dm_p130_fail, njets_p130_fail, indent="      ")

    print(f"     └─ has higher reco than matched      : {int(ak.sum(has_higher_fail))}")
    print_dm_jet_table("has higher reco than matched",
                       dm_fail_higher, njets_fail_higher, indent="        ")

    return {
        "matched_mask": matched_mask, "matched_gen_pt": matched_gen_pt,
        "matched_gen_eta": matched_gen_eta, "matched_gen_dm": matched_gen_dm,
        "matched_reco_pt": matched_reco_pt, "matched_best_dR": matched_best_dR,
        "after_gen_sel_gen_pt": after_gen_sel_gen_pt,
        "after_gen_sel_reco_pt": after_gen_sel_reco_pt,
        "after_gen_sel_dr": after_gen_sel_dr,
        "leading_reco": leading_reco,
    }


# ═══════════════════════════════════════════════════════════
# 4. gen=1 / gen=2 scan functions (with njets argument)
# ═══════════════════════════════════════════════════════════
def scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1,
                 gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                 njets_1,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5]):
    gen_pt_flat  = ak.flatten(gen_pt_1)
    gen_eta_flat = ak.flatten(gen_eta_1)
    gen_phi_flat = ak.flatten(gen_phi_1)
    gen_dm_flat  = ak.flatten(gen_dm_1)
    njets_arr    = np.asarray(njets_1, dtype=int)

    results = {}
    for dr in dr_list:
        results[dr] = analyze_tau_matching(
            reco_pt_1, reco_eta_1, reco_phi_1,
            gen_pt_flat, gen_eta_flat, gen_phi_flat, gen_dm_flat,
            njets_arr,
            DR_CUT=dr, label="Gen=1",
        )
    return results


def scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2,
                 gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                 njets_2,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5]):
    gen_sort_idx   = ak.argsort(gen_pt_2, axis=1, ascending=False)
    gen_pt_sorted  = gen_pt_2[gen_sort_idx]
    gen_eta_sorted = gen_eta_2[gen_sort_idx]
    gen_phi_sorted = gen_phi_2[gen_sort_idx]
    gen_dm_sorted  = gen_dm_2[gen_sort_idx]

    lead_pt  = ak.flatten(gen_pt_sorted[:, 0:1])
    lead_eta = ak.flatten(gen_eta_sorted[:, 0:1])
    lead_phi = ak.flatten(gen_phi_sorted[:, 0:1])
    lead_dm  = ak.flatten(gen_dm_sorted[:, 0:1])
    sub_pt   = ak.flatten(gen_pt_sorted[:, 1:2])
    sub_eta  = ak.flatten(gen_eta_sorted[:, 1:2])
    sub_phi  = ak.flatten(gen_phi_sorted[:, 1:2])
    sub_dm   = ak.flatten(gen_dm_sorted[:, 1:2])

    njets_arr = np.asarray(njets_2, dtype=int)

    results = {}
    for dr in dr_list:
        print(f"\n{'#'*60}\n  Gen=2  |  DR_CUT = {dr}\n{'#'*60}")

        gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(lead_eta[:, np.newaxis], reco_eta_2)
        gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(lead_phi[:, np.newaxis], reco_phi_2)
        deta = gen_eta_bc - reco_eta_bc
        dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
        best_dR_lead = ak.min(np.sqrt(deta**2 + dphi**2), axis=1)

        lead_matched_mask = best_dR_lead < dr
        lead_fail_mask    = ~lead_matched_mask
        lead_matched_np   = ak.to_numpy(ak.fill_none(lead_matched_mask, False)).astype(bool)
        lead_fail_np      = ~lead_matched_np

        print(f"\n  [Leading gen matched: {ak.sum(lead_matched_mask)} / {len(lead_pt)} events]")
        res_lead = analyze_tau_matching(
            reco_pt_2[lead_matched_mask], reco_eta_2[lead_matched_mask], reco_phi_2[lead_matched_mask],
            lead_pt[lead_matched_mask], lead_eta[lead_matched_mask],
            lead_phi[lead_matched_mask], lead_dm[lead_matched_mask],
            njets_arr[lead_matched_np],
            DR_CUT=dr, label="Gen=2 Leading matched",
        )

        print(f"\n  [Leading match failed → Trying subleading: {ak.sum(lead_fail_mask)} events]")
        res_sub = analyze_tau_matching(
            reco_pt_2[lead_fail_mask], reco_eta_2[lead_fail_mask], reco_phi_2[lead_fail_mask],
            sub_pt[lead_fail_mask], sub_eta[lead_fail_mask],
            sub_phi[lead_fail_mask], sub_dm[lead_fail_mask],
            njets_arr[lead_fail_np],
            DR_CUT=dr, label="Gen=2 Subleading (leading failed)",
        )
        results[dr] = {"leading": res_lead, "subleading": res_sub}
    return results


# ═══════════════════════════════════════════════════════════
# 5. Build selection-aligned GenVisTau / GenJet (η, φ) arrays
# ═══════════════════════════════════════════════════════════
genvis_eta_1 = genvis_eta[trigger_passed][mask2]
genvis_phi_1 = genvis_phi[trigger_passed][mask2]
genjet_eta_1 = genjet_eta[trigger_passed][mask2]
genjet_phi_1 = genjet_phi[trigger_passed][mask2]

genvis_eta_2 = genvis_eta[trigger_passed][mask3]
genvis_phi_2 = genvis_phi[trigger_passed][mask3]
genjet_eta_2 = genjet_eta[trigger_passed][mask3]
genjet_phi_2 = genjet_phi[trigger_passed][mask3]

# Gen=2 data (re-defined here so this cell can be run standalone)
gen_pt_2   = genvis_pt [trigger_passed][mask3]
gen_eta_2  = genvis_eta[trigger_passed][mask3]
gen_phi_2  = genvis_phi[trigger_passed][mask3]
gen_dm_2   = genvis_tau_dm[trigger_passed][mask3]
reco_pt_2  = tau_pt [trigger_passed][mask3]
reco_eta_2 = tau_eta[trigger_passed][mask3]
reco_phi_2 = tau_phi[trigger_passed][mask3]

# ═══════════════════════════════════════════════════════════
# 6. Isolated GenJet counts + sanity checks + run analysis
# ═══════════════════════════════════════════════════════════
njets_1 = count_isolated_jets(genvis_eta_1, genvis_phi_1,
                              genjet_eta_1, genjet_phi_1,
                              deltaR_threshold=0.3)
njets_2 = count_isolated_jets(genvis_eta_2, genvis_phi_2,
                              genjet_eta_2, genjet_phi_2,
                              deltaR_threshold=0.3)

assert len(njets_1) == len(gen_pt_1) == len(reco_pt_1), \
    f"gen=1 length mismatch: njets={len(njets_1)}, gen={len(gen_pt_1)}, reco={len(reco_pt_1)}"
assert len(njets_2) == len(gen_pt_2) == len(reco_pt_2), \
    f"gen=2 length mismatch: njets={len(njets_2)}, gen={len(gen_pt_2)}, reco={len(reco_pt_2)}"

res1 = scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1,
                    gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                    njets_1,
                    dr_list=[0.1])

res2 = scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2,
                    gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                    njets_2,
                    dr_list=[0.1])